# 02 — Data Cleaning of SentiPers

In this notebook, the SentiPers dataset is cleaned based on the results of the previous audit stage and prepared for modeling.

**This notebook covers:**

- Reloading the raw dataset
- Creating an independent cleaning copy
- Persian text normalization
- Detecting and removing invalid samples
- Handling same-label duplicate texts
- Removing conflicting label groups
- Final dataset quality validation
- Creating the cleaned dataset file
- Saving a complete cleaning report


## Data Cleaning List

- STEP 01 Imports and Paths
- STEP 02 Load Raw Data
- STEP 03 Define Persian Normalization
- STEP 04 Apply Text Normalization
- STEP 05 Remove Empty Texts
- STEP 06 Remove Same-Label Duplicates
- STEP 07 Remove Conflicting Labels
- STEP 08 Final Data Validation
- STEP 09 Save Cleaned Data
- STEP 10 Validate Saved Dataset
- Data Cleaning Conclusion


## STEP 01 — Import Libraries and Define Paths

Preparing required tools and defining input and output file paths.


In [51]:
import re
import unicodedata

import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 150)

DATA_PATH = Path("../data/raw/sentipers.xlsx")
PROCESSED_DATA_PATH = Path(
    "../data/processed/sentipers_clean_v1.csv"
)
CLEANING_REPORT_PATH = Path(
    "../outputs/tables/02_data_cleaning/cleaning_report.csv"
)

CLEANING_TABLES_DIR = Path(
    "../outputs/tables/02_data_cleaning"
)

CLEANING_TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

CLEANING_REPORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

print("Libraries loaded successfully.")
print(f"Raw data path: {DATA_PATH.resolve()}")
print(
    f"Processed data path: "
    f"{PROCESSED_DATA_PATH.resolve()}"
)
print(
    f"Cleaning report path: "
    f"{CLEANING_REPORT_PATH.resolve()}"
)

Libraries loaded successfully.
Raw data path: D:\Bachelor Project\PersianSentimentProject\data\raw\sentipers.xlsx
Processed data path: D:\Bachelor Project\PersianSentimentProject\data\processed\sentipers_clean_v1.csv
Cleaning report path: D:\Bachelor Project\PersianSentimentProject\outputs\tables\02_data_cleaning\cleaning_report.csv


## STEP 02 — Load Raw Dataset

Loading the original dataset and creating an independent copy for cleaning operations.


In [53]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {DATA_PATH.resolve()}"
    )

df_raw = pd.read_excel(DATA_PATH)

df_clean = df_raw.copy(deep=True)

INITIAL_ROW_COUNT = len(df_raw)
INITIAL_COLUMN_COUNT = df_raw.shape[1]

print("Raw dataset loaded successfully.")
print(f"Initial rows: {INITIAL_ROW_COUNT:,}")
print(f"Initial columns: {INITIAL_COLUMN_COUNT}")
print("Independent cleaning copy created.")

display(df_clean.head())

REQUIRED_COLUMNS = [
    "index",
    "sid",
    "text",
    "polarity",
    "file",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in df_clean.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if df_clean.empty:
    raise ValueError(
        "The dataset is empty."
    )

print("Required columns are available.")
print("The dataset is ready for cleaning.")

Raw dataset loaded successfully.
Initial rows: 15,683
Initial columns: 5
Independent cleaning copy created.


,index,sid,text,polarity,file
0,0,rev-1,اینک قصد داریم پرینتر دیگری از پرینترهای لیزری کمپانی Hp را معرفی کنیم.,0,data/main/HP LaserJet M1132.xml
1,1,rev-2,پرینتری چند کاره از رده‌ی Entry Level یا سطح مبتدی.,0,data/main/HP LaserJet M1132.xml
2,2,rev-3,به هر صورت اکنون ما در دنیایی زندگی می‌کنیم، که کاربران پرینترها انتظارات بالاتری علاوه بر گرفتن پرینت ساده از دستگاه خود دارند.,0,data/main/HP LaserJet M1132.xml
3,3,rev-4,به صورتی که توانایی کپی کردن، اسکن، فکس، پرینت عکس، پرینت دورو، قابلیت اتصال از طریق Bluetooth و WiFi را نیز باید داشته باشد.,0,data/main/HP LaserJet M1132.xml
4,4,rev-5,به هر صورت معمولا چیزی که بیشتر کاربران از پرینتری پر کار در این سطح قیمت برای خانه و یا دفتر کار انتظار دارند، تولید پرینت های با کیفیت بالا، ب...,2,data/main/HP LaserJet M1132.xml


Required columns are available.
The dataset is ready for cleaning.


## STEP 03 — Define Persian Text Normalization Function

Unifying Persian characters and spacing without changing the meaning of the text.


In [55]:
CHARACTER_TRANSLATION = str.maketrans(
    {
        "ي": "ی",
        "ى": "ی",
        "ك": "ک",
        "ۀ": "هٔ",
        "ؤ": "و",
        "إ": "ا",
        "أ": "ا",
        "ٱ": "ا",
    }
)

ARABIC_DIACRITICS_PATTERN = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]"
)

INVISIBLE_CHARACTERS_PATTERN = re.compile(
    r"[\u200B\u200D\u200E\u200F\u202A-\u202E\u2066-\u2069\uFEFF]"
)

MULTIPLE_SPACES_PATTERN = re.compile(r"[ \t]+")
MULTIPLE_NEWLINES_PATTERN = re.compile(r"\n+")

ZWNJ = "\u200c"


def normalize_persian_text(text):
    if pd.isna(text):
        return pd.NA

    normalized_text = str(text)

    normalized_text = unicodedata.normalize(
        "NFKC",
        normalized_text,
    )

    normalized_text = normalized_text.translate(
        CHARACTER_TRANSLATION
    )

    normalized_text = normalized_text.replace(
        "\u00A0",
        " ",
    )

    normalized_text = normalized_text.replace(
        "\u0640",
        "",
    )

    normalized_text = ARABIC_DIACRITICS_PATTERN.sub(
        "",
        normalized_text,
    )

    normalized_text = INVISIBLE_CHARACTERS_PATTERN.sub(
        "",
        normalized_text,
    )

    normalized_text = re.sub(
        rf"\s*{ZWNJ}\s*",
        ZWNJ,
        normalized_text,
    )

    normalized_text = re.sub(
        rf"{ZWNJ}+",
        ZWNJ,
        normalized_text,
    )

    normalized_text = MULTIPLE_SPACES_PATTERN.sub(
        " ",
        normalized_text,
    )

    normalized_text = MULTIPLE_NEWLINES_PATTERN.sub(
        " ",
        normalized_text,
    )

    return normalized_text.strip()


test_samples = [
    "اين يک متن فارسي است.",
    "محصول   بسيار     خوب بود.",
    "کیفیتِ محصول عالی بود.",
]

for sample in test_samples:
    print(f"Before: {sample}")
    print(f"After:  {normalize_persian_text(sample)}")
    print("-" * 60)

Before: اين يک متن فارسي است.
After:  این یک متن فارسی است.
------------------------------------------------------------
Before: محصول   بسيار     خوب بود.
After:  محصول بسیار خوب بود.
------------------------------------------------------------
Before: کیفیتِ محصول عالی بود.
After:  کیفیت محصول عالی بود.
------------------------------------------------------------


## STEP 04 — Apply Text Normalization

Applying the normalization function to dataset texts and analyzing the generated changes.


In [57]:
df_clean["text_original"] = (
    df_clean["text"]
    .astype("string")
)

df_clean["text"] = (
    df_clean["text_original"]
    .map(normalize_persian_text)
    .astype("string")
)

text_before_comparison = (
    df_clean["text_original"]
    .fillna("<NA>")
)

text_after_comparison = (
    df_clean["text"]
    .fillna("<NA>")
)

normalization_change_mask = (
    text_before_comparison
    .ne(text_after_comparison)
)

NORMALIZED_TEXT_CHANGE_COUNT = int(
    normalization_change_mask.sum()
)

normalization_preview = (
    df_clean.loc[
        normalization_change_mask,
        [
            "index",
            "text_original",
            "text",
        ],
    ]
    .rename(
        columns={
            "text_original": "before_normalization",
            "text": "after_normalization",
        }
    )
    .head(10)
    .reset_index(drop=True)
)

print("Text normalization completed.")
print(
    f"Normalized rows: "
    f"{NORMALIZED_TEXT_CHANGE_COUNT:,}"
)
print(
    f"Unchanged rows: "
    f"{len(df_clean) - NORMALIZED_TEXT_CHANGE_COUNT:,}"
)

display(normalization_preview)

Text normalization completed.
Normalized rows: 11,042
Unchanged rows: 4,641


,index,before_normalization,after_normalization
0,2,به هر صورت اکنون ما در دنیایی زندگی می‌کنیم، که کاربران پرینترها انتظارات بالاتری علاوه بر گرفتن پرینت ساده از دستگاه خود دارند.,به هر صورت اکنون ما در دنیایی زندگی می‌کنیم، که کاربران پرینترها انتظارات بالاتری علاوه بر گرفتن پرینت ساده از دستگاه خود دارند.
1,3,به صورتی که توانایی کپی کردن، اسکن، فکس، پرینت عکس، پرینت دورو، قابلیت اتصال از طریق Bluetooth و WiFi را نیز باید داشته باشد.,به صورتی که توانایی کپی کردن، اسکن، فکس، پرینت عکس، پرینت دورو، قابلیت اتصال از طریق Bluetooth و WiFi را نیز باید داشته باشد.
2,4,به هر صورت معمولا چیزی که بیشتر کاربران از پرینتری پر کار در این سطح قیمت برای خانه و یا دفتر کار انتظار دارند، تولید پرینت های با کیفیت بالا، ب...,به هر صورت معمولا چیزی که بیشتر کاربران از پرینتری پر کار در این سطح قیمت برای خانه و یا دفتر کار انتظار دارند، تولید پرینت های با کیفیت بالا، با ...
3,8,همانطور که گفتیم، M1132 پرینتری لیزری، تک رنگ و چند کاره است.,همانطور که گفتیم، M1132 پرینتری لیزری، تک رنگ و چند کاره است.
4,11,سینی جلویی توانایی نگهداری 150 برگ کاغذ A4 را دارد و درست در بالای آن، سینی خروجی قرار گرفته است که به صورت همزمان 100 کاغذ می‌توانند درون آن جای...,سینی جلویی توانایی نگهداری 150 برگ کاغذ A4 را دارد و درست در بالای آن، سینی خروجی قرار گرفته است که به صورت همزمان 100 کاغذ می‌توانند درون آن جای ...
5,14,علاوه بر این ها، اتصالات WiFi و Ethernet هم وجود ندارد.,علاوه بر این ها، اتصالات WiFi و Ethernet هم وجود ندارد.
6,16,Hp زمان راه اندازی و Setup را با استفاده از فن آوری Smart Install Technology به طور چشمگیری کاهش داده است که به شما اجازه می‌دهد تا بدون احتیاج به...,Hp زمان راه اندازی و Setup را با استفاده از فن آوری Smart Install Technology به طور چشمگیری کاهش داده است که به شما اجازه می‌دهد تا بدون احتیاج به...
7,17,پنل کنترل خیلی ساده است و فقط دو نمایشگر LED در کنار دکمه های بالا و پایین، Start و Stop، دکمه‌ی Setup، دکمه‌ی Dark/Light tone و دکمه‌ی Copy Si...,پنل کنترل خیلی ساده است و فقط دو نمایشگر LED در کنار دکمه های بالا و پایین، Start و Stop، دکمه‌ی Setup، دکمه‌ی Dark/Light tone و دکمه‌ی Copy Size ...
8,18,سرعت پرینت به نوعی فوق العاده می‌باشد، به طوری که در تنظیمات استاندارد با رزولوشن 600، این پرینتر توانایی چاپ 18 صفحه‌ی A4 در دقیقه را دارد.,سرعت پرینت به نوعی فوق العاده می‌باشد، به طوری که در تنظیمات استاندارد با رزولوشن 600، این پرینتر توانایی چاپ 18 صفحه‌ی A4 در دقیقه را دارد.
9,19,اگر در همین رزولوشن آن را بر روی Economic mode قرار دهید، سرعت به 19 صفحه در دقیقه افزایش می‌یابد.,اگر در همین رزولوشن آن را بر روی Economic mode قرار دهید، سرعت به 19 صفحه در دقیقه افزایش می‌یابد.


## STEP 05 — Remove Empty Text Samples

Rows whose text becomes empty after normalization are identified and removed from the cleaned dataset.


In [59]:
ROWS_BEFORE_EMPTY_REMOVAL = len(df_clean)

empty_text_mask = (
    df_clean["text"]
    .fillna("")
    .str.strip()
    .eq("")
)

removed_empty_text_rows = (
    df_clean.loc[empty_text_mask]
    .copy()
    .reset_index(drop=True)
)

df_clean = (
    df_clean.loc[~empty_text_mask]
    .copy()
    .reset_index(drop=True)
)

REMOVED_EMPTY_TEXT_COUNT = len(
    removed_empty_text_rows
)

ROWS_AFTER_EMPTY_REMOVAL = len(
    df_clean
)

empty_text_cleaning_summary = pd.DataFrame(
    {
        "metric": [
            "rows_before",
            "removed_empty_text_rows",
            "rows_after",
        ],
        "value": [
            ROWS_BEFORE_EMPTY_REMOVAL,
            REMOVED_EMPTY_TEXT_COUNT,
            ROWS_AFTER_EMPTY_REMOVAL,
        ],
    }
)

assert (
    ROWS_AFTER_EMPTY_REMOVAL
    == ROWS_BEFORE_EMPTY_REMOVAL
    - REMOVED_EMPTY_TEXT_COUNT
)

assert (
    df_clean["text"]
    .fillna("")
    .str.strip()
    .ne("")
    .all()
)

print(
    f"Rows before removal: "
    f"{ROWS_BEFORE_EMPTY_REMOVAL:,}"
)

print(
    f"Empty-text rows removed: "
    f"{REMOVED_EMPTY_TEXT_COUNT:,}"
)

print(
    f"Rows after removal: "
    f"{ROWS_AFTER_EMPTY_REMOVAL:,}"
)

display(
    empty_text_cleaning_summary
)

if REMOVED_EMPTY_TEXT_COUNT > 0:
    display(
        removed_empty_text_rows.head(10)
    )

Rows before removal: 15,683
Empty-text rows removed: 0
Rows after removal: 15,683


,metric,value
0,rows_before,15683
1,removed_empty_text_rows,0
2,rows_after,15683


## STEP 06 — Remove Same-Label Duplicates

Extra copies of normalized identical texts with the same sentiment label are removed.


In [61]:
ROWS_BEFORE_DUPLICATE_REMOVAL = len(
    df_clean
)

text_group_sizes = (
    df_clean
    .groupby("text")["text"]
    .transform("size")
)

text_group_label_counts = (
    df_clean
    .groupby("text")["polarity"]
    .transform("nunique")
)

same_label_duplicate_group_mask = (
    (text_group_sizes > 1)
    & (text_group_label_counts == 1)
)

SAME_LABEL_DUPLICATE_GROUP_COUNT = (
    df_clean.loc[
        same_label_duplicate_group_mask,
        "text",
    ]
    .nunique()
)

extra_duplicate_row_mask = (
    same_label_duplicate_group_mask
    & df_clean.duplicated(
        subset=["text"],
        keep="first",
    )
)

removed_same_label_duplicates = (
    df_clean.loc[
        extra_duplicate_row_mask
    ]
    .copy()
    .reset_index(drop=True)
)

df_clean = (
    df_clean.loc[
        ~extra_duplicate_row_mask
    ]
    .copy()
    .reset_index(drop=True)
)

REMOVED_SAME_LABEL_DUPLICATE_COUNT = len(
    removed_same_label_duplicates
)

ROWS_AFTER_DUPLICATE_REMOVAL = len(
    df_clean
)

duplicate_cleaning_summary = pd.DataFrame(
    {
        "metric": [
            "rows_before",
            "same_label_duplicate_groups",
            "removed_duplicate_rows",
            "rows_after",
        ],
        "value": [
            ROWS_BEFORE_DUPLICATE_REMOVAL,
            SAME_LABEL_DUPLICATE_GROUP_COUNT,
            REMOVED_SAME_LABEL_DUPLICATE_COUNT,
            ROWS_AFTER_DUPLICATE_REMOVAL,
        ],
    }
)

remaining_group_sizes = (
    df_clean
    .groupby("text")["text"]
    .transform("size")
)

remaining_group_label_counts = (
    df_clean
    .groupby("text")["polarity"]
    .transform("nunique")
)

remaining_same_label_duplicate_mask = (
    (remaining_group_sizes > 1)
    & (remaining_group_label_counts == 1)
)

assert (
    ROWS_AFTER_DUPLICATE_REMOVAL
    == ROWS_BEFORE_DUPLICATE_REMOVAL
    - REMOVED_SAME_LABEL_DUPLICATE_COUNT
)

assert not (
    remaining_same_label_duplicate_mask.any()
)

print(
    f"Rows before removal: "
    f"{ROWS_BEFORE_DUPLICATE_REMOVAL:,}"
)

print(
    f"Same-label duplicate groups: "
    f"{SAME_LABEL_DUPLICATE_GROUP_COUNT:,}"
)

print(
    f"Duplicate rows removed: "
    f"{REMOVED_SAME_LABEL_DUPLICATE_COUNT:,}"
)

print(
    f"Rows after removal: "
    f"{ROWS_AFTER_DUPLICATE_REMOVAL:,}"
)

display(
    duplicate_cleaning_summary
)

if REMOVED_SAME_LABEL_DUPLICATE_COUNT > 0:
    display(
        removed_same_label_duplicates.head(10)
    )

Rows before removal: 15,683
Same-label duplicate groups: 2,077
Duplicate rows removed: 2,154
Rows after removal: 13,529


,metric,value
0,rows_before,15683
1,same_label_duplicate_groups,2077
2,removed_duplicate_rows,2154
3,rows_after,13529


,index,sid,text,polarity,file,text_original
0,4,cr-5-5,در عوض رزولوشن بالای تصویر دقت تصاویر رو خیلی بالا برده و در آخر با پردازنده ی دو هسته ای این اسمارت فون شما قادر به انجام هر گونه بازی و مرور در ...,1,data/main/Sony Xperia Ion.xml,در عوض رزولوشن بالاي تصوير دقت تصاوير رو خيلي بالا برده و در آخر با پردازنده ي دو هسته اي اين اسمارت فون شما قادر به انجام هر گونه بازي و مرور در ...
1,1,gr-39-2,ممنون دیجی کالا,0,data/main/HP_ProBook_4540s-A.xml,ممنون ديجي کالا
2,18,cr-10-19,"""سپاس""",0,data/main/Apple iPhone 5 - 16GB.xml,"""سپاس"""
3,26,rev-27,خوب، Hero به طرز شگفت انگیزی کوچک و سبک وزن است.,1,data/main/GoPro HD Hero 960.xml,خوب، Hero به طرز شگفت انگیزی کوچک و سبک وزن است.
4,48,rev-49,پس از اینکه کوچک و سبک بودن، آسانی نحوه ی اتصال، حمل و نقل، خاصیت ضد آب و ضد ضربه ی این دوربین بوسیله ی محافظ پلی کربناتی برای ما ثابت شد، به سراغ...,1,data/main/GoPro HD Hero 960.xml,پس از اینکه کوچک و سبک بودن، آسانی نحوه ی اتصال، حمل و نقل، خاصیت ضد آب و ضد ضربه ی این دوربین بوسیله ی محافظ پلی کربناتی برای ما ثابت شد، به...
5,77,rev-78,بر روی بدنه یک پورت USB 2.0 برای اتصال به رایانه به منظور انتقال فایل ها و شارژ باتری ، خروجی HDMI برای اتصال به تلوزیون های دیجیتال و جک 2.5 میلی...,0,data/main/GoPro HD Hero 960.xml,بر روی بدنه یک پورت USB 2.0 برای اتصال به رایانه به منظور انتقال فایل ها و شارژ باتری ، خروجی HDMI برای اتصال به تلوزیون های دیجیتال و جک 2.5 میل...
6,1,gr-7-2,واقعا محشره,2,data/main/Samsung Galaxy S III I9305 - 16GB.xml,واقعاً محشره\n
7,45,cr-6-46,این کنترل از نسخه‌ی اولیه‌ی آن که در کنسول XBOX 360 نسخه‌ی Premium وجود داشت، بزرگ‌تر است و از لحاظ امکانات و قابلیت‌ها، از نسخه‌ی قبلی خود کامل‌ت...,1,data/main/Microsoft Xbox 360 Elite.xml,اين کنترل از نسخه‌ي اوليه‌ي آن که در کنسول XBOX 360 نسخه‌ي Premium وجود داشت، بزرگ ‌تر است و از لحاظ امکانات و قابليت‌ها، از نسخه‌ي قبلي خود کامل‌...
8,55,rev-56,دوربین اصلی این گوشی دارای یک سنسور 8 مگاپیکسلی می‌باشد که قابلیت فیلم برداری با کیفیت FullHD با سرعت 30 فریم بر ثانیه را نیز داراست.,0,data/main/LG Optimus Vu P895.xml,دوربین اصلی این گوشی دارای یک سنسور 8 مگاپیکسلی می‌باشد که قابلیت فیلم برداری با کیفیت FullHD با سرعت 30 فریم بر ثانیه را نیز داراست.
9,16,rev-17,محیط نرم افزاری به لطف پردازنده بسیار قدرتمند دو هسته ای و حافظه RAM بالا، بسیار روان است و می‌توانیم بگوییم که به جز حالتی که از تصویر پس زمینه م...,2,data/main/Samsung Galaxy Tab 2 10.1 P5100 - 16GB.xml,محیط نرم افزاری به لطف پردازنده بسیار قدرتمند دو هسته ای و حافظه RAM بالا، بسیار روان است و می‌توانیم بگوییم که به جز حالتی که از تصویر پس زمینه ...


## STEP 07 — Remove Conflicting Label Groups

Text groups assigned with multiple sentiment labels are identified and removed from the cleaned dataset.


In [63]:
ROWS_BEFORE_CONFLICT_REMOVAL = len(
    df_clean
)

text_label_counts = (
    df_clean
    .groupby("text")["polarity"]
    .transform("nunique")
)

conflicting_label_mask = (
    text_label_counts > 1
)

CONFLICTING_TEXT_GROUP_COUNT = (
    df_clean.loc[
        conflicting_label_mask,
        "text",
    ]
    .nunique()
)

removed_conflicting_label_rows = (
    df_clean.loc[
        conflicting_label_mask
    ]
    .copy()
    .sort_values(
        [
            "text",
            "polarity",
        ]
    )
    .reset_index(drop=True)
)

df_clean = (
    df_clean.loc[
        ~conflicting_label_mask
    ]
    .copy()
    .reset_index(drop=True)
)

REMOVED_CONFLICTING_ROW_COUNT = len(
    removed_conflicting_label_rows
)

ROWS_AFTER_CONFLICT_REMOVAL = len(
    df_clean
)

conflict_cleaning_summary = pd.DataFrame(
    {
        "metric": [
            "rows_before",
            "conflicting_text_groups",
            "removed_conflicting_rows",
            "rows_after",
        ],
        "value": [
            ROWS_BEFORE_CONFLICT_REMOVAL,
            CONFLICTING_TEXT_GROUP_COUNT,
            REMOVED_CONFLICTING_ROW_COUNT,
            ROWS_AFTER_CONFLICT_REMOVAL,
        ],
    }
)

remaining_label_counts = (
    df_clean
    .groupby("text")["polarity"]
    .nunique()
)

assert (
    ROWS_AFTER_CONFLICT_REMOVAL
    == ROWS_BEFORE_CONFLICT_REMOVAL
    - REMOVED_CONFLICTING_ROW_COUNT
)

assert (
    remaining_label_counts
    .le(1)
    .all()
)

print(
    f"Rows before removal: "
    f"{ROWS_BEFORE_CONFLICT_REMOVAL:,}"
)

print(
    f"Conflicting text groups: "
    f"{CONFLICTING_TEXT_GROUP_COUNT:,}"
)

print(
    f"Conflicting rows removed: "
    f"{REMOVED_CONFLICTING_ROW_COUNT:,}"
)

print(
    f"Rows after removal: "
    f"{ROWS_AFTER_CONFLICT_REMOVAL:,}"
)

display(
    conflict_cleaning_summary
)

if REMOVED_CONFLICTING_ROW_COUNT > 0:
    display(
        removed_conflicting_label_rows.head(10)
    )

Rows before removal: 13,529
Conflicting text groups: 198
Conflicting rows removed: 435
Rows after removal: 13,094


,metric,value
0,rows_before,13529
1,conflicting_text_groups,198
2,removed_conflicting_rows,435
3,rows_after,13094


,index,sid,text,polarity,file,text_original
0,11380,-,) جای قرار گرفتن فلاشه که اولین عکسی که میخوای بگیری فلاش باز میشه و میاد بالا که باید حواست باشه به انگشت اشاره دست چپت برخورد نکنه.,-1,data/extra/Sony Cyber-Shot DSC-HX10V.xml,) جاي قرار گرفتن فلاشه که اولين عکسي که ميخواي بگيري فلاش باز ميشه و مياد بالا که بايد حواست باشه به انگشت اشاره دست چپت برخورد نکنه.
1,4,gr-1-5,) جای قرار گرفتن فلاشه که اولین عکسی که میخوای بگیری فلاش باز میشه و میاد بالا که باید حواست باشه به انگشت اشاره دست چپت برخورد نکنه.,0,data/main/Sony Cyber-Shot DSC-HX10V.xml,) جاي قرار گرفتن فلاشه که اولين عکسي که ميخواي بگيري فلاش باز ميشه و مياد بالا که بايد حواست باشه به انگشت اشاره دست چپت برخورد نکنه.
2,11548,-,N43SM در کانفیگ‌هایی با ظرفیت‌ها‌ی هارد دیسک مختلف نیز ساخته شده است و از حافظه‌ها‌ی SSD نیز پشتیبانی می‌کند.,0,data/extra/Asus N43SM-C_Hosseini.xml,N43SM در کانفیگ‌هایی با ظرفیت‌ها‌ی هارد دیسک مختلف نیز ساخته شده است و از حافظه‌ها‌ی SSD نیز پشتیبانی می‌کند.
3,21,rev-22,N43SM در کانفیگ‌هایی با ظرفیت‌ها‌ی هارد دیسک مختلف نیز ساخته شده است و از حافظه‌ها‌ی SSD نیز پشتیبانی می‌کند.,1,data/main/Asus N43SM-C.xml,N43SM در کانفیگ‌هایی با ظرفیت‌ها‌ی هارد دیسک مختلف نیز ساخته شده است و از حافظه‌ها‌ی SSD نیز پشتیبانی می‌کند.
4,1,gr-1-2,one x plus تنها گوشی بود که منو مجذوب خودش کرد:و اما دلیلش: 1.,1,data/main/HTC One X Plus - 64GB.xml,one x plus تنها گوشي بود که منو مجذوب خودش کرد:و اما دليلش:\n1.
5,3677,-,one x plus تنها گوشی بود که منو مجذوب خودش کرد:و اما دلیلش: 1.,2,data/extra/HTC One X Plus - 64GB.xml,one x plus تنها گوشي بود که منو مجذوب خودش کرد:و اما دليلش: 1.
6,11319,-,آهنگ و از همه مهم تر استفاده از فلش کار رو خیلی راحت و دستگاه رو خیلی ساده کرده به همه دوستان توصیه می کنم,1,data/extra/Samsung Galaxy Tab 2 10.1 P5100 - 16GB.xml,آهنگ و از همه مهم تر استفاده از فلش کار رو خيلي راحت و دستگاه رو خيلي ساده کرده به همه دوستان توصيه مي کنم
7,5,gr-9-6,آهنگ و از همه مهم تر استفاده از فلش کار رو خیلی راحت و دستگاه رو خیلی ساده کرده به همه دوستان توصیه می کنم,2,data/main/Samsung Galaxy Tab 2 10.1 P5100 - 16GB.xml,آهنگ و از همه مهم تر استفاده از فلش کار رو خيلي راحت و دستگاه رو خيلي ساده کرده\nبه همه دوستان توصيه مي کنم \n
8,66,rev-67,آیفون 5 در رده دوم ایستاد و آیفون 4S نیز سوم شد، هر سه از اپل !,0,data/main/Apple iPhone 5 - 16GB.xml,آیفون 5 در رده دوم ایستاد و آیفون 4S نیز سوم شد، هر سه از اپل !
9,1264,-,آیفون 5 در رده دوم ایستاد و آیفون 4S نیز سوم شد، هر سه از اپل !,0,data/extra/Apple iPhone 5 - 16GB_Hosseini.xml,آیفون 5 در رده دوم ایستاد و آیفون 4S نیز سوم شد، هر سه از اپل !


## STEP 08 — Final Cleaning Audit

The cleaned dataset is reviewed after all cleaning operations to verify data quality and consistency.


In [65]:
final_validation_checks = {
    "no_missing_text": (
        df_clean["text"]
        .notna()
        .all()
    ),
    "no_blank_text": (
        df_clean["text"]
        .fillna("")
        .str.strip()
        .ne("")
        .all()
    ),
    "no_missing_labels": (
        df_clean["polarity"]
        .notna()
        .all()
    ),
    "unique_normalized_texts": (
        df_clean["text"]
        .is_unique
    ),
}

final_validation_summary = pd.DataFrame(
    {
        "check": list(
            final_validation_checks.keys()
        ),
        "passed": list(
            final_validation_checks.values()
        ),
    }
)

original_class_distribution = (
    df_raw["polarity"]
    .value_counts()
    .sort_index()
    .rename("before_cleaning")
)

clean_class_distribution = (
    df_clean["polarity"]
    .value_counts()
    .sort_index()
    .rename("after_cleaning")
)

class_distribution_comparison = (
    pd.concat(
        [
            original_class_distribution,
            clean_class_distribution,
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)

class_distribution_comparison[
    "removed_rows"
] = (
    class_distribution_comparison[
        "before_cleaning"
    ]
    - class_distribution_comparison[
        "after_cleaning"
    ]
)

class_distribution_comparison[
    "remaining_percentage"
] = (
    class_distribution_comparison[
        "after_cleaning"
    ]
    .div(
        class_distribution_comparison[
            "before_cleaning"
        ]
        .replace(0, pd.NA)
    )
    .mul(100)
    .round(2)
)

class_distribution_comparison.index.name = (
    "polarity"
)

class_distribution_comparison = (
    class_distribution_comparison
    .reset_index()
)

TOTAL_ORIGINAL_ROWS = len(
    df_raw
)

TOTAL_CLEAN_ROWS = len(
    df_clean
)

TOTAL_REMOVED_ROWS = (
    TOTAL_ORIGINAL_ROWS
    - TOTAL_CLEAN_ROWS
)

REMAINING_DATA_PERCENTAGE = round(
    (
        TOTAL_CLEAN_ROWS
        / TOTAL_ORIGINAL_ROWS
    )
    * 100,
    2,
)

final_cleaning_summary = pd.DataFrame(
    {
        "metric": [
            "original_rows",
            "clean_rows",
            "total_removed_rows",
            "remaining_data_percentage",
        ],
        "value": [
            TOTAL_ORIGINAL_ROWS,
            TOTAL_CLEAN_ROWS,
            TOTAL_REMOVED_ROWS,
            REMAINING_DATA_PERCENTAGE,
        ],
    }
)

assert all(
    final_validation_checks.values()
)

assert (
    TOTAL_CLEAN_ROWS
    == ROWS_AFTER_CONFLICT_REMOVAL
)

assert (
    TOTAL_REMOVED_ROWS
    >= 0
)

print(
    f"Original rows: "
    f"{TOTAL_ORIGINAL_ROWS:,}"
)

print(
    f"Clean rows: "
    f"{TOTAL_CLEAN_ROWS:,}"
)

print(
    f"Total removed rows: "
    f"{TOTAL_REMOVED_ROWS:,}"
)

print(
    f"Remaining data: "
    f"{REMAINING_DATA_PERCENTAGE:.2f}%"
)

print(
    "All final validation checks passed."
)

display(
    final_validation_summary
)

display(
    final_cleaning_summary
)

display(
    class_distribution_comparison
)

Original rows: 15,683
Clean rows: 13,094
Total removed rows: 2,589
Remaining data: 83.49%
All final validation checks passed.


,check,passed
0,no_missing_text,True
1,no_blank_text,True
2,no_missing_labels,True
3,unique_normalized_texts,True


,metric,value
0,original_rows,15683.00
1,clean_rows,13094.00
2,total_removed_rows,2589.00
3,remaining_data_percentage,83.49


,polarity,before_cleaning,after_cleaning,removed_rows,remaining_percentage
0,-2,212,184,28,86.79
1,-1,1574,1376,198,87.42
2,0,5938,4959,979,83.51
3,1,5019,4172,847,83.12
4,2,2940,2403,537,81.73


## STEP 09 — Generate Cleaning Report

A complete report of all cleaning operations and removed samples is generated.


In [78]:
PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

CLEANING_TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

cleaning_report = pd.DataFrame(
    [
        {
            "operation": "empty_text_removal",
            "rows_before": ROWS_BEFORE_EMPTY_REMOVAL,
            "affected_groups": REMOVED_EMPTY_TEXT_COUNT,
            "removed_rows": REMOVED_EMPTY_TEXT_COUNT,
            "rows_after": ROWS_AFTER_EMPTY_REMOVAL,
        },
        {
            "operation": "same_label_duplicate_removal",
            "rows_before": ROWS_BEFORE_DUPLICATE_REMOVAL,
            "affected_groups": SAME_LABEL_DUPLICATE_GROUP_COUNT,
            "removed_rows": REMOVED_SAME_LABEL_DUPLICATE_COUNT,
            "rows_after": ROWS_AFTER_DUPLICATE_REMOVAL,
        },
        {
            "operation": "conflicting_label_removal",
            "rows_before": ROWS_BEFORE_CONFLICT_REMOVAL,
            "affected_groups": CONFLICTING_TEXT_GROUP_COUNT,
            "removed_rows": REMOVED_CONFLICTING_ROW_COUNT,
            "rows_after": ROWS_AFTER_CONFLICT_REMOVAL,
        },
    ]
)

overall_cleaning_result = pd.DataFrame(
    [
        {
            "operation": "overall_cleaning",
            "rows_before": TOTAL_ORIGINAL_ROWS,
            "affected_groups": pd.NA,
            "removed_rows": TOTAL_REMOVED_ROWS,
            "rows_after": TOTAL_CLEAN_ROWS,
        }
    ]
)

cleaning_report = pd.concat(
    [
        cleaning_report,
        overall_cleaning_result,
    ],
    ignore_index=True,
)

FINAL_DATA_COLUMNS = [
    "index",
    "sid",
    "text",
    "polarity",
    "file",
]

missing_final_columns = [
    column
    for column in FINAL_DATA_COLUMNS
    if column not in df_clean.columns
]

if missing_final_columns:
    raise ValueError(
        f"Missing final columns: "
        f"{missing_final_columns}"
    )

final_dataset = (
    df_clean[
        FINAL_DATA_COLUMNS
    ]
    .copy()
    .reset_index(drop=True)
)

final_dataset.to_csv(
    PROCESSED_DATA_PATH,
    index=False,
    encoding="utf-8-sig",
)

report_tables = {
    "cleaning_report.csv": cleaning_report,
    "final_cleaning_summary.csv": final_cleaning_summary,
    "cleaning_validation.csv": final_validation_summary,
    "class_distribution_comparison.csv": (
        class_distribution_comparison
    ),
    "removed_empty_text_rows.csv": (
        removed_empty_text_rows
    ),
    "removed_same_label_duplicates.csv": (
        removed_same_label_duplicates
    ),
    "removed_conflicting_label_rows.csv": (
        removed_conflicting_label_rows
    ),
}

saved_report_paths = []

for filename, table in report_tables.items():
    output_path = (
        CLEANING_TABLES_DIR
        / filename
    )

    table.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    saved_report_paths.append(
        output_path
    )

assert PROCESSED_DATA_PATH.is_file()

assert all(
    path.is_file()
    for path in saved_report_paths
)

assert (
    len(final_dataset)
    == TOTAL_CLEAN_ROWS
)   

print(
    f"Clean dataset rows: "
    f"{len(final_dataset):,}"
)

print(
    f"Clean dataset saved to: "
    f"{PROCESSED_DATA_PATH.resolve()}"
)

print(
    f"Report files saved: "
    f"{len(saved_report_paths):,}"
)

print(
    f"Reports saved to: "
    f"{CLEANING_TABLES_DIR.resolve()}"
)

display(
    cleaning_report
)

Clean dataset rows: 13,094
Clean dataset saved to: D:\Bachelor Project\PersianSentimentProject\data\processed\sentipers_clean_v1.csv
Report files saved: 7
Reports saved to: D:\Bachelor Project\PersianSentimentProject\outputs\tables\02_data_cleaning


,operation,rows_before,affected_groups,removed_rows,rows_after
0,empty_text_removal,15683,0,0,15683
1,same_label_duplicate_removal,15683,2077,2154,13529
2,conflicting_label_removal,13529,198,435,13094
3,overall_cleaning,15683,<NA>,2589,13094


## STEP 10 — Final Dataset Verification and Saving

The cleaned dataset is validated and saved as the final input file for modeling stages.


In [81]:
assert PROCESSED_DATA_PATH.is_file()

reloaded_clean_df = pd.read_csv(
    PROCESSED_DATA_PATH,
    encoding="utf-8-sig",
)

expected_clean_df = (
    final_dataset
    .reset_index(drop=True)
)

reloaded_clean_df = (
    reloaded_clean_df
    .reset_index(drop=True)
)

ROWS_MATCH = (
    len(reloaded_clean_df)
    == len(expected_clean_df)
)

COLUMNS_MATCH = (
    list(reloaded_clean_df.columns)
    == list(expected_clean_df.columns)
)

try:
    pd.testing.assert_frame_equal(
        reloaded_clean_df,
        expected_clean_df,
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12,
    )

    CONTENT_MATCHES = True

except AssertionError:
    CONTENT_MATCHES = False

NO_MISSING_TEXT = (
    reloaded_clean_df["text"]
    .notna()
    .all()
)

NO_BLANK_TEXT = (
    reloaded_clean_df["text"]
    .fillna("")
    .str.strip()
    .ne("")
    .all()
)

NO_MISSING_LABELS = (
    reloaded_clean_df["polarity"]
    .notna()
    .all()
)

UNIQUE_TEXTS = (
    reloaded_clean_df["text"]
    .is_unique
)

EXPECTED_CLASS_DISTRIBUTION = (
    expected_clean_df["polarity"]
    .value_counts()
    .sort_index()
)

RELOADED_CLASS_DISTRIBUTION = (
    reloaded_clean_df["polarity"]
    .value_counts()
    .sort_index()
)

CLASS_DISTRIBUTION_MATCHES = (
    EXPECTED_CLASS_DISTRIBUTION
    .equals(
        RELOADED_CLASS_DISTRIBUTION
    )
)

saved_dataset_checks = {
    "file_exists": (
        PROCESSED_DATA_PATH.is_file()
    ),
    "rows_match": ROWS_MATCH,
    "columns_match": COLUMNS_MATCH,
    "content_matches": CONTENT_MATCHES,
    "no_missing_text": NO_MISSING_TEXT,
    "no_blank_text": NO_BLANK_TEXT,
    "no_missing_labels": NO_MISSING_LABELS,
    "unique_texts": UNIQUE_TEXTS,
    "class_distribution_matches": (
        CLASS_DISTRIBUTION_MATCHES
    ),
}

saved_dataset_validation = pd.DataFrame(
    {
        "check": list(
            saved_dataset_checks.keys()
        ),
        "passed": list(
            saved_dataset_checks.values()
        ),
    }
)

VALIDATION_REPORT_PATH = (
    CLEANING_TABLES_DIR
    / "saved_dataset_validation.csv"
)

saved_dataset_validation.to_csv(
    VALIDATION_REPORT_PATH,
    index=False,
    encoding="utf-8-sig",
)

assert all(
    saved_dataset_checks.values()
)

assert (
    len(reloaded_clean_df)
    == TOTAL_CLEAN_ROWS
)

assert VALIDATION_REPORT_PATH.is_file()

print(
    f"Reloaded rows: "
    f"{len(reloaded_clean_df):,}"
)

print(
    f"Reloaded columns: "
    f"{reloaded_clean_df.shape[1]:,}"
)

print(
    f"Dataset path: "
    f"{PROCESSED_DATA_PATH.resolve()}"
)

print(
    f"Validation report: "
    f"{VALIDATION_REPORT_PATH.resolve()}"
)

print(
    "All saved dataset checks passed."
)

display(
    saved_dataset_validation
)

display(
    reloaded_clean_df.head()
)

Reloaded rows: 13,094
Reloaded columns: 5
Dataset path: D:\Bachelor Project\PersianSentimentProject\data\processed\sentipers_clean_v1.csv
Validation report: D:\Bachelor Project\PersianSentimentProject\outputs\tables\02_data_cleaning\saved_dataset_validation.csv
All saved dataset checks passed.


,check,passed
0,file_exists,True
1,rows_match,True
2,columns_match,True
3,content_matches,True
4,no_missing_text,True
5,no_blank_text,True
6,no_missing_labels,True
7,unique_texts,True
8,class_distribution_matches,True


,index,sid,text,polarity,file
0,0,rev-1,اینک قصد داریم پرینتر دیگری از پرینترهای لیزری کمپانی Hp را معرفی کنیم.,0,data/main/HP LaserJet M1132.xml
1,1,rev-2,پرینتری چند کاره از رده‌ی Entry Level یا سطح مبتدی.,0,data/main/HP LaserJet M1132.xml
2,2,rev-3,به هر صورت اکنون ما در دنیایی زندگی می‌کنیم، که کاربران پرینترها انتظارات بالاتری علاوه بر گرفتن پرینت ساده از دستگاه خود دارند.,0,data/main/HP LaserJet M1132.xml
3,3,rev-4,به صورتی که توانایی کپی کردن، اسکن، فکس، پرینت عکس، پرینت دورو، قابلیت اتصال از طریق Bluetooth و WiFi را نیز باید داشته باشد.,0,data/main/HP LaserJet M1132.xml
4,4,rev-5,به هر صورت معمولا چیزی که بیشتر کاربران از پرینتری پر کار در این سطح قیمت برای خانه و یا دفتر کار انتظار دارند، تولید پرینت های با کیفیت بالا، با ...,2,data/main/HP LaserJet M1132.xml


## Conclusion — Final Clean Dataset Preparation

In this notebook, the SentiPers dataset was cleaned based on the previous audit results. Text normalization, invalid sample removal, duplicate handling, and conflicting label removal were performed while keeping the original raw dataset unchanged.

**Next step:** The cleaned dataset is now ready for feature extraction, classic machine learning models, and ParsBERT Fine-tuning.
